In [13]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Polygon, Circle
from matplotlib.path import Path
import math
import ast



def xsit_value(pos_balon, portero, jugadores):
    fig, ax = plt.subplots(figsize=(10, 6), dpi=100)
    ax.set_xlim(0, 120)
    ax.set_ylim(0, 75)
    ax.set_xticks([])  
    ax.set_yticks([])  
    ax.set_frame_on(False)
    ax.set_facecolor('white')

    # Portería
    porteria = [(120, 32), (120, 43)] if pos_balon[0] >= 60 else [(0, 32), (0, 43)]
    
    # Triángulo
    vertices_triangulo = np.array([pos_balon, porteria[0], porteria[1]])
    triangulo = Polygon(vertices_triangulo, closed=True, edgecolor='red', facecolor=(1, 0, 0), linewidth=2)
    ax.add_patch(triangulo)

    # Balón
    ax.scatter(*pos_balon, color='red', s=100, label="Ball")

    # Jugadores con radios dinámicos
    for jugador in jugadores:
        circulo = Circle(jugador, radius=1.5, color='blue', alpha=0.5)
        ax.add_patch(circulo)

    # Portero con radio dinámico
    por = Circle(portero, radius=2, color='green', alpha=0.5, label="Goalkeeper")
    ax.add_patch(por)

    plt.legend()

    # Procesamiento de la imagen para calcular el área blanca
    fig.canvas.draw()
    w, h = fig.canvas.get_width_height()
    image = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8').reshape((h, w, 3))
    plt.close(fig)

    transform = ax.transData
    pix_vertices = transform.transform(vertices_triangulo)
    pix_vertices[:, 1] = h - pix_vertices[:, 1]

    path = Path(pix_vertices)
    Y, X = np.mgrid[0:h, 0:w]
    coords = np.vstack((X.ravel(), Y.ravel())).T
    mask = path.contains_points(coords).reshape((h, w))

    colores_dentro = image[mask]
    colores_redondeados = (colores_dentro // 10) * 10
    colores_unicos, counts = np.unique(colores_redondeados, axis=0, return_counts=True)

    for color, count in zip(colores_unicos, counts):
        if tuple(color) == (250, 0, 0):
            porcentaje = count / np.sum(counts)
            return porcentaje

def calcular_velocidad(df):
    """
    Añade una columna 'xSIT' al DataFrame calculando la zona libre visible en portería.
    Elimina las filas que generan errores durante el procesamiento.
    """
    xsit_values = []
    indices_a_eliminar = []
    
    for idx, row in df.iterrows():
        try:
            # Convertir strings a listas si es necesario
            loc = row['location']
            loc = ast.literal_eval(loc)

            players_location = row['players_location']
            players_location = ast.literal_eval(players_location)
            gk_location = row['goalkeeper_location']
            gk_location = ast.literal_eval(gk_location)

            xsit = xsit_value(loc, gk_location, players_location)
            xsit_values.append(xsit)
        except Exception as e:
            indices_a_eliminar.append(idx)
    
    # Eliminar filas con errores
    df = df.drop(indices_a_eliminar)
    
    # Asignar valores xSIT solo a las filas que no tuvieron errores
    df['xSIT'] = xsit_values
    
    return df

def procesar_excel(archivo_entrada, archivo_salida=None):
    """
    Procesa el archivo Excel, añadiendo la columna de velocidad.
    Si no se especifica archivo_salida, sobreescribe el original.
    """
    # Leer el archivo Excel
    df = pd.read_excel(archivo_entrada)
    
    # Calcular velocidades
    df = calcular_velocidad(df)
    
    # Guardar el resultado
    if archivo_salida is None:
        archivo_salida = archivo_entrada
    
    df.to_excel(archivo_salida, index=False)
    print(f"Archivo guardado en: {archivo_salida}")

archivo_original = "C:/Users/Usuario/Desktop/Investigacion_blanca/open-data-master/open-data-master/data/events/72/velocidad_con_calculo.xlsx"
archivo_resultado = "C:/Users/Usuario/Desktop/Investigacion_blanca/open-data-master/open-data-master/data/events/72/masculino_completo.xlsx"
    
procesar_excel(archivo_original, archivo_resultado)

C:\Users\Usuario\AppData\Local\Temp\ipykernel_20052\224611362.py:45: MatplotlibDeprecationWarning: The tostring_rgb function was deprecated in Matplotlib 3.8 and will be removed two minor releases later. Use buffer_rgba instead.
  image = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8').reshape((h, w, 3))
C:\Users\Usuario\AppData\Local\Temp\ipykernel_20052\224611362.py:45: MatplotlibDeprecationWarning: The tostring_rgb function was deprecated in Matplotlib 3.8 and will be removed two minor releases later. Use buffer_rgba instead.
  image = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8').reshape((h, w, 3))


PermissionError: [Errno 13] Permission denied: 'C:/Users/Usuario/Desktop/Investigacion_blanca/open-data-master/open-data-master/data/events/72/masculino_completo.xlsx'

Con velocidad

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Polygon, Circle
from matplotlib.path import Path
import math
import ast

def calcular_tiempo_llegada(pos_balon, velocidad_balon, pos_jugador):
    """Calcula el tiempo que tarda el balón en llegar a la posición del jugador"""
    distancia = math.sqrt((pos_balon[0] - pos_jugador[0])**2 + (pos_balon[1] - pos_jugador[1])**2)
    tiempo = distancia / (velocidad_balon * 1000/3600)  # Convertir km/h a m/s
    return tiempo

def calcular_radio_efectivo(tiempo_llegada, radio_base=1.5, tiempo_reaccion=0.4):
    """Ajusta el radio en función del tiempo de llegada vs tiempo de reacción"""
    factor = tiempo_llegada / tiempo_reaccion
    return radio_base * factor

def xsitv_value(pos_balon, portero, jugadores, velocidad_balon):
    fig, ax = plt.subplots(figsize=(10, 6), dpi=100)
    ax.set_xlim(0, 120)
    ax.set_ylim(0, 75)
    ax.set_xticks([])  
    ax.set_yticks([])  
    ax.set_frame_on(False)
    ax.set_facecolor('white')

    # Portería
    porteria = [(120, 32), (120, 43)] if pos_balon[0] >= 60 else [(0, 32), (0, 43)]
    
    # Triángulo
    vertices_triangulo = np.array([pos_balon, porteria[0], porteria[1]])
    triangulo = Polygon(vertices_triangulo, closed=True, edgecolor='red', facecolor=(1, 0, 0), linewidth=2)
    ax.add_patch(triangulo)

    # Balón
    ax.scatter(*pos_balon, color='red', s=100, label="Ball")

    # Jugadores con radios dinámicos
    for jugador in jugadores:
        tiempo = calcular_tiempo_llegada(pos_balon, velocidad_balon, jugador)
        radio = calcular_radio_efectivo(tiempo, radio_base=1.5)
        circulo = Circle(jugador, radius=radio, color='blue', alpha=0.5)
        ax.add_patch(circulo)

    # Portero con radio dinámico
    tiempo_portero = calcular_tiempo_llegada(pos_balon, velocidad_balon, portero)
    radio_portero = calcular_radio_efectivo(tiempo_portero, radio_base=2.0)
    por = Circle(portero, radius=radio_portero, color='green', alpha=0.5, label="Goalkeeper")
    ax.add_patch(por)

    plt.legend()

    # Procesamiento de la imagen para calcular el área blanca
    fig.canvas.draw()
    w, h = fig.canvas.get_width_height()
    image = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8').reshape((h, w, 3))
    plt.close(fig)

    transform = ax.transData
    pix_vertices = transform.transform(vertices_triangulo)
    pix_vertices[:, 1] = h - pix_vertices[:, 1]

    path = Path(pix_vertices)
    Y, X = np.mgrid[0:h, 0:w]
    coords = np.vstack((X.ravel(), Y.ravel())).T
    mask = path.contains_points(coords).reshape((h, w))

    colores_dentro = image[mask]
    colores_redondeados = (colores_dentro // 10) * 10
    colores_unicos, counts = np.unique(colores_redondeados, axis=0, return_counts=True)

    for color, count in zip(colores_unicos, counts):
        if tuple(color) == (250, 0, 0):
            porcentaje = count / np.sum(counts)
            return porcentaje

def calcular_velocidad(df):
    """
    Añade una columna 'xSIT' al DataFrame calculando la zona libre visible en portería.
    Elimina las filas que generan errores durante el procesamiento.
    """
    xsit_values = []
    indices_a_eliminar = []
    
    for idx, row in df.iterrows():
        try:
            # Convertir strings a listas si es necesario
            loc = row['location']
            loc = ast.literal_eval(loc)

            players_location = row['players_location']
            players_location = ast.literal_eval(players_location)
            gk_location = row['goalkeeper_location']
            gk_location = ast.literal_eval(gk_location)
            
            velocidad = row['velocidad']

            xsit = xsitv_value(loc, gk_location, players_location, velocidad)
            xsit_values.append(xsit)
        except Exception as e:
            indices_a_eliminar.append(idx)
    
    # Eliminar filas con errores
    df = df.drop(indices_a_eliminar)
    
    # Asignar valores xSIT solo a las filas que no tuvieron errores
    df['xSIT'] = xsit_values
    
    return df

def procesar_excel(archivo_entrada, archivo_salida=None):
    """
    Procesa el archivo Excel, añadiendo la columna de velocidad.
    Si no se especifica archivo_salida, sobreescribe el original.
    """
    # Leer el archivo Excel
    df = pd.read_excel(archivo_entrada)
    
    # Calcular velocidades
    df = calcular_velocidad(df)
    
    # Guardar el resultado
    if archivo_salida is None:
        archivo_salida = archivo_entrada
    
    df.to_excel(archivo_salida, index=False)
    print(f"Archivo guardado en: {archivo_salida}")

archivo_original = "C:/Users/Usuario/Desktop/Investigacion_blanca/open-data-master/open-data-master/data/events/43/velocidad_con_calculo.xlsx"
archivo_resultado = "C:/Users/Usuario/Desktop/Investigacion_blanca/open-data-master/open-data-master/data/events/43/masculino_completo_velocidad.xlsx"
    
procesar_excel(archivo_original, archivo_resultado)

C:\Users\Usuario\AppData\Local\Temp\ipykernel_20052\2779689636.py:58: MatplotlibDeprecationWarning: The tostring_rgb function was deprecated in Matplotlib 3.8 and will be removed two minor releases later. Use buffer_rgba instead.
  image = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8').reshape((h, w, 3))


Archivo guardado en: C:/Users/Usuario/Desktop/Investigacion_blanca/open-data-master/open-data-master/data/events/43/masculino_completo_velocidad.xlsx
